### Profitable App Profiles for the App Store and Google Play Markets

#### About: 

_This project leverages the concepts learned from the <ins>Part 1 - Python Introduction | Section A. Python Fundamentals</ins>._

__Scenario:__ We're working as data analysts for a company that builds Android and iOS mobile apps. We make our apps available on Google Play and the App Store. We only build apps that are free to download and install, and our main source of revenue consists of in-app ads. This means our revenue for any given app is mostly influenced by the number of users who use our app — the more users that see and engage with the ads, the better.

__Goal:__ To analyze data to help developers understand what type of apps are likely to attract more users on Google Play and the App Store.

#### Exploring Data

As of September 2018, there were approximately 2 million iOS apps available on the App Store, and 2.1 million Android apps on Google Play.

- A <a href = 'https://www.kaggle.com/lava18/google-play-store-apps'>data set</a> containing data about approximately 10,000 Android apps from Google Play; the data was collected in August 2018. You can download the data set directly from <a href = 'https://dq-content.s3.amazonaws.com/350/googleplaystore.csv'> this link</a>.
- A <a href = 'https://www.kaggle.com/ramamet4/app-store-apple-data-set-10k-apps'>data set</a> containing data about approximately 7,000 iOS apps from the App Store; the data was collected in July 2017. You can download the data set directly from <a href='https://dq-content.s3.amazonaws.com/350/AppleStore.csv'> this link</a>.

In [ ]:
from csv import reader

# Apple Store (IOS) Apps data
ios_all_data = list(reader(open('data/AppleStore.csv', encoding='utf8')))
ios_header = ios_all_data[0]
ios_data = ios_all_data[1:]

# Google Play Store (Android) Apps data
android_all_data = list(reader(open('data/googleplaystore.csv', encoding='utf8')))
android_header = android_all_data[0]
android_data = android_all_data[1:]

In [ ]:
def explore_data(dataset, start, end, rows_and_columns = False):
    dataset_slice = dataset[start:end]
    for row in dataset_slice:
        print(row)
        print('\n')
        
    if rows_and_columns:
        print('Number of rows: ', len(dataset))
        print('Number of columns: ', len(dataset[0]))

##### IOS dataset exploration

In [ ]:
print(ios_header)
print('\n')
explore_data(ios_data, 2, 6, True)

#### **_We noticed that the Apple Store contains 7197 mobile apps and Every app has 16 different types of information associated with it._**

The columns that could help in our analysis are: 'track_name', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'prime_genre'

Not all of the column names are self-explanatory. In order to know more about this dataset, check out <a href= 'https://www.kaggle.com/ramamet4/app-store-apple-data-set-10k-apps'> this link</a>

##### Android dataset exploration

In [ ]:
print(android_header)
print('\n')
explore_data(android_data, 2, 6, True)

#### **_We noticed that the Google Play Store contains 10841 mobile apps and Every app has 13 different types of information associated with it._**

The columns of interest are: 'App', 'Category', 'Reviews', 'Installs', 'Type', 'Price', 'Genres'

#### Data cleaning

- Remove non-English Apps
- Remove apps that aren't free

This process of preparing the data for analysis is called __data cleaning__. Data cleaning is done before the analysis; it includes removing or correcting wrong data, removing duplicate data, and modifying the data to fit the purpose of the analysis.

The Google Play data set has a dedicated <a href='https://www.kaggle.com/lava18/google-play-store-apps/discussion'> discussion section</a>, and <a href= 'https://www.kaggle.com/lava18/google-play-store-apps/discussion/66015'> one of the discussions</a> describes an error for a row 10472.

In [ ]:
print(android_data[10472])
print('\n')
print(android_header)

Seems like row [10472] has a missing entry for the __'Category'__ column which shifted the results for the rest of the columns. So, remove this row from the dataset.

__Note: DO NOT RUN the _del_ command MORE THAN ONCE. Otherwise, you'll end up LOSING DATA__

In [ ]:
del android_data[10472]

Read the <a href='https://www.kaggle.com/ramamet4/app-store-apple-data-set-10k-apps/discussion'> discussion section</a> for the App Store data set, and see whether there are any reports of wrong data.

To look for the same type of error as the Google Dataset, use the code below to check if there exists any missing values by comapring the length of the rows to the header.

In [ ]:
for row in ios_data:
    if len(row) != len(ios_header):
        print(ios_data.index(row), row)

##### Removing Duplicate rows: Part One

After exploring the Google Play data set long enough or looking at the <a href= 'https://www.kaggle.com/lava18/google-play-store-apps/discussion'> discussions</a> section, notice some apps have duplicate entries. For instance, Instagram has four entries:

In [ ]:
for app in android_data:
    name = app[0]
    if name == 'Instagram':
        print(app)

In [ ]:
duplicate_apps = []
unique_apps = []

for app in android_data:
    name = app[0]
    if name in unique_apps:
        duplicate_apps.append(name)
    else:
        unique_apps.append(name)
        
print('Number of duplicate apps: ', len(duplicate_apps))
print('\n')
print('Number of unique apps: ', len(unique_apps))

In total, there are 1,181 cases where an app occurs more than once:

##### List few duplicate entries

In [ ]:
print('List of a few duplicate apps:', duplicate_apps[:10])

**_Note_** that we don't want to remove duplicates at random. Taking the example of Instagram, we noticed that the one entry that changes is the rating counts.

So, we can keep the highest count as it is the most recent. We will use this criterion to filter duplicates

To remove the duplicates, we will:

- Create a dictionary, where each dictionary key is a unique app name and the corresponding dictionary value is the highest number of reviews of that app.
- Use the information stored in the dictionary and create a new data set, which will have only one entry per app (and for each app, we'll only select the entry with the highest number of reviews).

##### Removing Duplicate Rows: Part Two

1. Create a dictionary where each key is a unique app name and the corresponding dictionary value is the highest number of reviews of that app.

    - Start by creating an empty dictionary named reviews_max.
    - Loop through the Google Play data set (make sure you don't include the header row). For each iteration:
        - Assign the app name to a variable named name.
        - Convert the number of reviews to float. Assign it to a variable named n_reviews.
        - If name already exists as a key in the reviews_max dictionary and reviews_max[name] < n_reviews, update the number of reviews for that entry in the reviews_max dictionary.
        - If name is not in the reviews_max dictionary as a key, create a new entry in the dictionary where the key is the app name, and the value is the number of reviews. Make sure you don't use an else clause here, otherwise the number of reviews will be incorrectly updated whenever reviews_max[name] < n_reviews evaluates to False.
    - Inspect the dictionary to make sure everything went as expected. Measure the length of the dictionary - remember that the expected length is 9,659 entries.

In [ ]:
reviews_max={}
for app in android_data:
    name = app[0]
    n_reviews = float(app[3])
    if name in reviews_max and (reviews_max[name] < n_reviews):
        reviews_max[name] = n_reviews
    if name not in reviews_max:
        reviews_max[name] = n_reviews
print(len(reviews_max))

2. Use the dictionary you created above to remove the duplicate rows:

    - Start by creating two empty lists: android_clean (which will store our new cleaned data set) and already_added (which will just store app names).
    - Loop through the Google Play data set (make sure you don't include the header row), and for each iteration:
        - Assign the app name to a variable named name.
        - Convert the number of reviews to float, and assign it to a variable named n_reviews.
    - If n_reviews is the same as the number of maximum reviews of the app name (the number can be found in the reviews_max dictionary) and name is not already in the list already_added (read the solution notebook to find out why we need this supplementary condition):
        - Append the entire row to the android_clean list (which will eventually be a list of list and store our cleaned data set).
        - Append the name of the app name to the already_added list - this helps us to keep track of apps that we already added.

In [ ]:
android_clean = []
already_added = []

for app in android_data:
    name = app[0]
    n_reviews = float(app[3])
    if (n_reviews == reviews_max[name]) and (name not in already_added):
        android_clean.append(app)
        already_added.append(name)

Let's now explore the '__android_clean__' dataset to ensure everything went as expected. The dataset should have 9,659 rows

In [ ]:
explore_data(android_clean, 0, 4, True)

Great! We have 9659 rows, as expected

##### Removing Non-English Apps: Part One

If we explore the data long enough, we'll find that both data sets have apps with names that suggest they are not directed toward an English-speaking audience.

In [ ]:
print(ios_data[813][1])
print(ios_data[6731][1])
print('\n')
print(android_clean[4412][0])
print(android_clean[7940][0])

We're not interested in keeping these apps, so we'll remove them. One way to go about this is to remove each app with a name containing a symbol that is not commonly used in English text — English text usually includes letters from the English alphabet, numbers composed of digits from 0 to 9, punctuation marks (., !, ?, ;), and other symbols (+, *, /).

All English characters range from 0 to 127, according to the ASCII (American Standard Code for Information Interchange) system. So, if we build a function that checks if any character fall within this range to qualify as a common English character.

If an app name contains a character that is greater than 127, then it probably means that the app has a non-English name. Our app names, however, are stored as strings, so how could we take each individual character of a string and check its corresponding number?

We can get the corresponding number of each character using the `ord()` built-in function.

1. Write a function that takes in a string and returns `False` if there's any character in the string that doesn't belong to the set of common English characters, otherwise it returns `True`.

    - Inside the function, iterate over the input string. For each iteration check whether the number associated with the character is greater than 127. When a character is greater than 127, the function should immediately return `False` — the app name is probably non-English since it contains a character that doesn't belong to the set of common English characters.
    - If the loop finishes running without the return statement being executed, then it means no character had a corresponding number over 127 — the app name is probably English, so the functions should return `True`.

In [ ]:
def is_english(a_string):
    for char in a_string:
        if ord(char) > 127:
            return False
    return True 

2. Use your function to check whether these app names are detected as English or non-English:

- 'Instagram'
- '爱奇艺PPS -《欢乐颂2》电视剧热播'
- 'Docs To Go™ Free Office Suite'
- 'Instachat 😜'

In [ ]:
print(is_english('Instagram'))
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播'))
print(is_english('Docs To Go™ Free Office Suite'))
print(is_english('Instachat 😜'))

The functions seems to miscalculate strings with emoji or other symbols such as (™,—(em dash), -(en dash)) that fall outside the ASCII range.

So, our function is not fully ready to run on the actual dataset. Let's look at ways to modify it so we do not lose useful information.

In [ ]:
print(ord('™'))
print(ord('😜'))

##### Part Two

To minimize the impact of data loss, we'll only remove an app if its name has more than three characters with corresponding numbers falling outside the ASCII range.

This means all English apps with up to three emoji or other special characters will still be labeled as English. Our filter function is still not perfect, but it should be fairly effective.

1. If the input string has more than three characters that fall outside the ASCII range (0 - 127), then the function should return False (identify the string as non-English), otherwise it should return True.

In [ ]:
def is_english(a_string):
    non_ascii_count = 0
    for char in a_string:
        if ord(char) > 127:
            non_ascii_count += 1  
    if non_ascii_count > 3:
        return False
    else:
        return True

2. Use the new function to check whether these app names are detected as English or non-English:

- 'Docs To Go™ Free Office Suite'
- 'Instachat 😜'
- '爱奇艺PPS -《欢乐颂2》电视剧热播'

In [ ]:
print(is_english('Docs To Go™ Free Office Suite'))
print(is_english('Instachat 😜'))
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播'))

3. Use the new function to filter out non-English apps from both data sets. Loop through each data set. If an app name is identified as English, append the whole row to a separate list.

- Our Current IOS dataset:  __ios_data__
- Our Current ANdroid dataset:  __android_clean__

In [ ]:
english_apps_ios = []
english_apps_android = []

# For the IOS data set
for row in ios_data:
    name = row[1]
    if is_english(name):
        english_apps_ios.append(row)

# For the Android data set
for row in android_clean:
    name = row[0]
    if is_english(name):
        english_apps_android.append(row)

4. Explore the data sets and see how many rows you have remaining for each data set.

In [ ]:
explore_data(english_apps_ios, 0, 3, True) # IOS Dataset

Our initial set of IOS App data had 7197 rows. After removing non-English apps, we are left with 6183 apps. 

In [ ]:
# The Android data
explore_data(english_apps_android, 0, 3, True)

Our android_clean dataset had 9659 rows. Our new English-only dataset consists of 9614 rows.

##### Isolating Free Apps

So far in the data cleaning process, we:

1. Removed inaccurate data
2. Removed duplicate app entries
3. Removed non-English apps

Our goal is to build apps that are free to download and install, and our main source of revenue comes from in-app ads. So, we need to isolate the free apps for our analysis.

Isolating the free apps will be our last step in the data cleaning process.

- Our Current IOS dataset:  __english_apps_ios__
- Our Current Android dataset:  __english_apps_android__

Note: The '__price__' column in the Android dataset has the "$$  " attached to the number in the form `'$4.99'`. In this case, we can either use the '__Type__' column (index 6) or remove the '$' from the price. Either way works

In [ ]:
free_apps_android = []
free_apps_ios = []

# The IOS dataset
for row in english_apps_ios:
    price = float(row[4])
    if price == 0:
        free_apps_ios.append(row)
        
# The Android dataset
for row in english_apps_android:
    price = float(row[7].strip('$')) # Or you can use the 'Type' column (index 6) whose results are 'Paid or Free'
    if price == 0:
        free_apps_android.append(row)

Let's explore the datasets now to check how many rows are we left with.

In [ ]:
# The IOS dataset
explore_data(free_apps_ios, 0, 3, True)

In [ ]:
# The Android dataset
explore_data(free_apps_android, 0, 3, True)

We are now left with __3222__ IOS Apps and __8864__ Android Apps

#### Most Common Apps by Genre: Part One

So far, we spent a good amount of time on cleaning data, and:

1. Removed inaccurate data
2. Removed duplicate app entries
3. Removed non-English apps
4. Isolated the free apps

Our aim is to determine the kinds of apps that are likely to attract more users because our revenue is highly influenced by the number of people using our apps.

To minimize risks and overhead, our validation strategy for an app idea is comprised of three steps:

1. Build a minimal Android version of the app, and add it to Google Play.
2. If the app has a good response from users, we develop it further.
3. If the app is profitable after six months, we build an iOS version of the app and add it to the App Store.

We need to find app profiles that are successful on both markets. For instance, a profile that works well for both markets might be a productivity app that makes use of gamification.

Let's begin the analysis by getting a sense of what are the most common genres for each market. For this, we'll need to build frequency tables for a few columns in our data sets.

For the __IOS__ dataset, we will use the '__prime_genre__' column to build freq tables.
For the __Android__ dataset, we will use the '__Genres__' and '__Category__' columns.

#### Part Two

We'll build two functions we can use to analyze the frequency tables:

- One function to generate frequency tables that show percentages
- Another function we can use to display the percentages in a descending order

In [ ]:
def freq_table(dataset, index):
    freq_table ={}
    total_apps = 0
    for row in dataset:
        total_apps += 1
        genre = row[index]
        if genre in freq_table:
            freq_table[genre] += 1
        else:
            freq_table[genre] = 1
    
    freq_table_percentages = {}
    
    for app in freq_table:
        percentage = (freq_table[app]/ total_apps) * 100
        freq_table_percentages[app] = percentage
        
    return freq_table_percentages

def display_table(dataset, index):
    table = freq_table(dataset, index)
    table_display = []
    for key in table:
        key_val_as_tuple = (table[key], key)
        table_display.append(key_val_as_tuple)

    table_sorted = sorted(table_display, reverse = True)
    for entry in table_sorted:
        print(entry[1], ':', entry[0])

#### Part Three

Display the frequency table of the columns prime_genre, Genres, and Category.

- Our Current IOS dataset:  __free_apps_ios__
- Our Current Android dataset:  __free_apps_android__

In [ ]:
# The IOS dataset with the prime_genre column (index 11)
display_table(free_apps_ios, 11)

Upon analyzing the IOS dataset, we noticed that 58.2% apps are games (most common) which is more than half of the dataset. 7.8% apps belong to the Entertainment genre (the runner-up) followed by Photo & Video with 4.9%. Only 3.66% apps are desihned for Education and Social Networking Apps consists of 3.28% of the entire dataset.

The general impression of the Apple Store apps, that are free and target the English speaking audience, is that they focus highly on apps for the purpose of entertainment such as Games, Photo and Video, Social Networking, sports, music etc. Practical apps such as education, shopping, utilities, productivity, lifestyle etc. are not as popular.

However, we can't solely rely on this frequency table to recommend an app profile. If a particular genre has numerous apps, that doesn't imply that these apps have a large number of users

In [ ]:
# The Android dataset with the Category column (index 1)
display_table(free_apps_android, 1)

The Google Play Store seems to have a pretty balanced app collection with apps for fun (game) as well as for practical purposes (tools, lifestyle, business). Even that Family category is the most popular, upon checking out, we observed that it consists of games mostly for kids

In [ ]:
# The Android dataset with the genres column (index 9)
display_table(free_apps_android, 9)

The App Store is more Entertainment oriented while Google Play Store has more balanced set of apps. The frequency table for genres gives a more granular representation of the distribution of apps which seemed more balanced as opposed to the App Store.

We still can't use the frequency tables alone. So, let's check out the total users for each category or genre of apps to see if we can find a better app profile for our analysis.

#### Most Popular Apps by Genre on the App Store

One way to find out what genres are the most popular (have the most users) is to calculate the average number of installs for each app genre.

For the __Google Play__ data set, we can find this information in the **_Installs_** column, but this information is missing for the __App Store__ data set. As a workaround, we'll take the total number of user ratings as a proxy, which we can find in the **_rating_count_tot_** app.

- Our Current IOS dataset:  __free_apps_ios__
- Our Current Android dataset:  __free_apps_android__

__Let's start with calculating the average number of user ratings per app genre on the App Store__

In [ ]:
ios_prime_genre_ft = freq_table(free_apps_ios, 11)

for genre in ios_prime_genre_ft:
    total = 0 # This variable will store the sum of user ratings (the number of ratings, not the actual ratings) specific to each genre.
    len_genre = 0 # This variable will store the number of apps specific to each genre.
    for app in free_apps_ios:
        genre_app = app[11]
        if genre_app == genre:
            n_ratings = float(app[5])
            total += n_ratings
            len_genre += 1
    avg_ratings = total / len_genre
    print(genre, ':', avg_ratings)

Upon analysis, Navigation Apps have the highest user ratings with an average of 86090.33. Social Network also are popular with third highest average user ratings.

In [ ]:
for app in free_apps_ios:
    if app[11] == 'Navigation':
        print(app[1], ': ', app[5]) # Name and total ratings

Even though Navigation is at the top, most of their ratings is driven by two specific apps, namely, Waze and Google Maps which almost half a million ratings.

Now Let's take a look at the Social Networking Apps:

In [ ]:
for app in free_apps_ios:
    if app[11] == 'Social Networking':
        print(app[1], ': ', app[5]) # Name and total ratings

Similar to Navigation Apps, SN is heavily dominated by a few popular apps such as Facebook,Pinterest,Skype for iPhone, Messenger(Facebook owned), Tumblr & WhatsApp Messenger(also owned by FB).

Even though we have these top genres, it is still not enough to give a proper App profile. With a few popular apps dominating an entire genre makes it harder for other apps to make the 10000 threshold. We could eliminate these popular apps for each genre and recalculate the average.

Let's look at the Reference list as well:

In [ ]:
for app in free_apps_ios:
    if app[11] == 'Reference':
        print(app[1], ': ', app[5]) # Name and total ratings

Reference rank second in the list with 74942.11 user ratings on average. However, the results are skewed mostly by two apps, Bible and Dictionary.com. However, this genre is more practical than for fun which seems to be domainting the App Store. Developing an app for a book and adding different features such as a dictionary could stand a chance.

Other genres that are popular are as follows:
1. Music : 57326.53
2. Weather : 52279.89
3. Book : 39758.5
4. Finance : 31467.94

The book genre can be interesting for us, the rest doesn't fuel to our App profile recommendation.

__Let's take a look at the Google Play Store Apps:__

We have data about the number of installs for the Google Play market, so we should be able to get a clearer picture about genre popularity. However, the install numbers don't seem precise enough — we can see that most values are open-ended (100+, 1,000+, 5,000+, etc.)

In [ ]:
display_table(free_apps_android, 5) # The Installs column

As we can see that our data is not precise. For instance, we don't know whether an app with 100,000+ installs has 100,000 installs, 200,000, or 350,000. Since we only want to find out which app genres attract the most users, we don't need perfect precision with respect to the number of users.

We're going to leave the numbers as they are, which means that we'll consider that an app with 100,000+ installs has 100,000 installs, and an app with 1,000,000+ installs has 1,000,000 installs, and so on.

To perform computations, however, we'll need to convert each install number from string to float. This means we need to remove the commas and the plus characters, otherwise the conversion will fail and raise an error.

- Our Current Android dataset: __free_apps_android__

In [ ]:
android_category_ft = freq_table(free_apps_android, 1)

for category in android_category_ft:
    total = 0         # This variable will store the sum of installs specific to each genre.
    len_category = 0  # This variable will store the number of apps specific to each genre.
    for app in free_apps_android:
        category_app = app[1]
        if category_app == category:
            n_installs = app[5]
            n_installs = n_installs.replace('+', '')
            n_installs = n_installs.replace(',', '')
            total += float(n_installs)
            len_category += 1
    avg_installs = total / len_category
    print(category, ':', avg_installs)

On average, Communication Apps have the highest number of installs (38,456,119). Upon deeper analysis, we noticed that only a few popular apps are skewing the results just like the ones we observed in the App Store dataset. Apps such as WhatsApp, Messenger, Skype, Google Apps(Google Chrome, Gmail, Hangouts) alone has over a billion installs among other apps.

In [ ]:
# Communication Category
for app in free_apps_android:
    if app[1] == 'COMMUNICATION' and (app[5] == '100,000,000+' or app[5] == '500,000,000+' or app[5] == '1,000,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

If we are to remove these apps with over 100 million downloads, we see that their average is reduced ten times

In [ ]:
under_100_mil = []

for app in free_apps_android:
    n_installs = app[5]
    n_installs = n_installs.replace(',','')
    n_installs = n_installs.replace('+','')
    if (app[1] == 'COMMUNICATION') and (float(n_installs) < 100000000):
        under_100_mil.append(float(n_installs))

sum(under_100_mil) / len(under_100_mil)

For the Video Player category, we have an average installs of 24727872. We see the same pattern here where a few apps dominate the market such as YouTube, Google Play Movies & TV or MX Player. The pattern is same for Social Apps (Facebook, Google+, Instagram, etc.) and Photography Apps (Google Photos, B612, Sweet Selfie, etc.).

In [ ]:
# Video Players Category
for app in free_apps_android:
    if app[1] == 'VIDEO_PLAYERS' and (app[5] == '100,000,000+' or app[5] == '500,000,000+' or app[5] == '1,000,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

In [ ]:
# Social App Category
for app in free_apps_android:
    if app[1] == 'SOCIAL' and (app[5] == '100,000,000+' or app[5] == '500,000,000+' or app[5] == '1,000,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

In [ ]:
# Photography Category
for app in free_apps_android:
    if app[1] == 'PHOTOGRAPHY' and (app[5] == '100,000,000+' or app[5] == '500,000,000+' or app[5] == '1,000,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

Our main goal is to build an app which can dominate both the markets. For App Store, we recommended Book App and we see that BOOK & REFERENCE Category is also quite popular in the Google Play Store with an average installs of 8767811. Maybe we can explore this category a little deeper.

In [ ]:
# Book & Reference category
for app in free_apps_android:
    if app[1] == 'BOOKS_AND_REFERENCE':
        print(app[0], ':', app[5])

Let's filter them to get a list of the most installed Apps.

In [ ]:
# Book & Reference Category
for app in free_apps_android:
    if app[1] == 'BOOKS_AND_REFERENCE' and (app[5] == '100,000,000+' or app[5] == '500,000,000+' or app[5] == '1,000,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

This list is too small so we could lower our threshold to see if more apps show up.

In [ ]:
# Book & Reference Category
for app in free_apps_android:
    if app[1] == 'BOOKS_AND_REFERENCE' and (app[5] == '1,000,000+' or app[5] == '5,000,000+' or app[5] == '10,000,000+' or app[5] == '50,000,000+'):
        print(app[0], ':', app[5]) # Name and Number of Installs

There exists a lot of apps that processes and reads ebooks, as well as dictionaries and various libraries for translation. So, building a similar app will increase competition which may not be helpful for us.

We also see a lot of apps about the Quran, which suggests that maybe an app for a popular book can be profitable for us for both Apple Store and Google Play markets.

However, it looks like the market is already full of libraries, so we need to add some special features besides the raw version of the book. This might include daily quotes from the book, an audio version of the book, quizzes on the book, a forum where people can discuss the book, etc.

### Conclusions

In this project, we analyzed two app datasets from the Google Play store and Apple Store respectively. We cleaned the dataset to remove duplicate and wrong data and focused on Apps that are free to download and targets an English speaking audience.

We identified the most popular apps based on genre and user ratings or user installs and found that most of these most apps are dominated by one or two highly popular ones which skewed the results significantly. So, we focused on selecting an app that is more practical and not just for fun.

We concluded that taking a very popular book and turning it into an app could be profitable for both the Google Play and the App Store market. The markets are already full of libraries, so we need to add some special features besides the raw version of the book. This might include daily quotes from the book, an audio version of the book, quizzes on the book, a forum where people can discuss the book, etc.